# 04 — Baseline & Model Comparison

**Goal:** establish an honest baseline, then compare model families under
identical cross-validation before spending compute on tuning.

**Candidates and why:**

| Model | Role |
|---|---|
| Logistic Regression | interpretable linear baseline — any complex model must beat it |
| Random Forest | non-linear interactions, robust default |
| XGBoost | gradient boosting — usually the strongest on tabular data |

**Class imbalance** (~26.5% churners) is handled by cost weighting
(`class_weight='balanced'` / `scale_pos_weight`) rather than resampling —
simpler, leakage-free, and equivalent in effect for these models.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

In [2]:
from src.feature_engineering import build_preprocessor, get_feature_lists
from src.model_training import compare_models, get_models
from src.utils import load_config

config = load_config()
train = pd.read_csv(ROOT / config["data"]["train_path"])
X_train, y_train = train.drop(columns=["churn_value"]), train["churn_value"]

numeric, categorical = get_feature_lists(X_train)
preprocessor = build_preprocessor(numeric, categorical)
models = get_models(y_train, config["training"]["random_state"])

## 5-fold stratified cross-validation

Every pipeline (preprocessor + model) is cross-validated as a unit, so scaling
and encoding are re-fitted inside each fold — cross-validating only the model
on pre-transformed data is a classic leakage bug.

Primary comparison metric: **ROC-AUC** (threshold-independent ranking
quality). Recall/precision at the default 0.5 threshold are shown for context
only — the operating threshold is tuned later.

In [3]:
comparison = compare_models(models, preprocessor, X_train, y_train, config)
comparison

2026-07-31 09:39:39,827 | src.model_training | INFO | CV logistic_regression: ROC-AUC=0.8589 recall=0.8114


2026-07-31 09:39:42,543 | src.model_training | INFO | CV random_forest: ROC-AUC=0.8461 recall=0.6763


2026-07-31 09:39:45,893 | src.model_training | INFO | CV xgboost: ROC-AUC=0.8396 recall=0.6649


,model,roc_auc,avg_precision,recall,precision,f1
0,logistic_regression,0.858921,0.682563,0.811371,0.532125,0.642498
1,random_forest,0.846084,0.640447,0.676254,0.586245,0.627894
2,xgboost,0.839581,0.640365,0.664883,0.567153,0.611954


## Hyperparameter tuning

The winning family is tuned with **RandomizedSearchCV** (40 candidates × 5
folds) in `src/train_pipeline.py` — random search covers wide spaces at a
fixed compute budget far better than grid search. The tuned results:

In [4]:
import json

with open(ROOT / config["artifacts"]["metrics_path"]) as fh:
    metrics = json.load(fh)

print(f"best model:  {metrics['model']}")
print(f"CV ROC-AUC:  {metrics['cv_roc_auc']:.4f}")
print("best params:")
for k, v in metrics["best_params"].items():
    print(f"  {k.removeprefix('model__')}: {v}")

best model:  logistic_regression
CV ROC-AUC:  0.8590
best params:
  C: 2.6373339933815254


> Full reproducible run: `python -m src.train_pipeline` — trains, tunes,
> evaluates and serialises the final artifact end to end.